# Antibody Data Sources

Working code for accessing TheraSAbDab (positives) and OAS (negatives).

In [ ]:
import gzip
import io
import json
import re

import pandas as pd
import requests

## 1. TheraSAbDab (Positives)

Therapeutic antibodies — approved + clinical trials.

In [ ]:
THERASABDAB_URL = (
    "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/static/downloads/"
    "TheraSAbDab_SeqStruc_OnlineDownload.csv"
)

resp = requests.get(THERASABDAB_URL, timeout=60)
resp.raise_for_status()
thera_df = pd.read_csv(io.StringIO(resp.text))
print(f"TheraSAbDab: {len(thera_df)} entries")
thera_df.head(3)

In [ ]:
# Extract VH sequences
vh_col = "HeavySequence"
therapeutic_vh = thera_df[thera_df[vh_col].notna() & (thera_df[vh_col] != "na")][vh_col].drop_duplicates()
print(f"Unique therapeutic VH: {len(therapeutic_vh)}")

## 2. OAS (Negatives)

General human antibodies from NGS repertoire studies.

In [ ]:
# Get download URLs for human Heavy chain data
oas_resp = requests.post(
    "https://opig.stats.ox.ac.uk/webapps/oas/oas_unpaired/",
    data={"Species": "human", "Chain": "Heavy"},
    timeout=60,
)

# Parse wget URLs from response
oas_urls = re.findall(r'wget (https://[^"]+\.csv\.gz)', oas_resp.text)
print(f"Available OAS data units: {len(oas_urls)}")

In [ ]:
def download_oas_unit(url: str) -> tuple[dict, pd.DataFrame]:
    """Download single OAS data unit. Returns (metadata, sequences_df)."""
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    content = gzip.decompress(resp.content).decode("utf-8")
    lines = content.strip().split("\n")
    metadata = json.loads(lines[0])
    df = pd.read_csv(io.StringIO("\n".join(lines[1:])))
    return metadata, df

In [ ]:
# Download one unit as example
meta, oas_sample = download_oas_unit(oas_urls[0])
print(f"Metadata: {meta['Author']}, {meta['Species']}, {meta['BType']}")
print(f"Sequences: {len(oas_sample)}")
oas_sample[["sequence", "productive", "v_call"]].head(3)

In [ ]:
def sample_oas_sequences(urls: list[str], n_target: int = 2000, seed: int = 42) -> list[str]:
    """Sample productive sequences from OAS data units."""
    import random
    random.seed(seed)
    random.shuffle(urls)
    
    sequences = set()
    for url in urls:
        if len(sequences) >= n_target:
            break
        try:
            _, df = download_oas_unit(url)
            productive = df[df["productive"] == "T"]["sequence"].dropna().tolist()
            sequences.update(productive[:500])  # Cap per unit
            print(f"  {len(sequences)}/{n_target} sequences", end="\r")
        except Exception as e:
            continue
    print()
    return list(sequences)[:n_target]

In [ ]:
# Sample 100 sequences (demo — increase for real use)
oas_sequences = sample_oas_sequences(oas_urls[:10], n_target=100)
print(f"Sampled {len(oas_sequences)} OAS sequences")

## 3. Combine Dataset

In [ ]:
# Remove any OAS sequences that match therapeutics
therapeutic_set = set(therapeutic_vh)
oas_clean = [s for s in oas_sequences if s not in therapeutic_set]
print(f"OAS after removing therapeutic overlap: {len(oas_clean)}")

In [ ]:
# Create combined dataset
positives = pd.DataFrame({"sequence": therapeutic_vh.tolist(), "label": 1, "source": "therasabdab"})
negatives = pd.DataFrame({"sequence": oas_clean, "label": 0, "source": "oas"})
combined = pd.concat([positives, negatives], ignore_index=True)

print(f"Combined: {len(combined)} ({combined['label'].sum()} pos, {len(combined) - combined['label'].sum()} neg)")

## Summary

| Source | Sequences | Access |
|--------|-----------|--------|
| TheraSAbDab | ~1000 therapeutic VH | Direct CSV download |
| OAS | 2.4B+ (sample as needed) | wget .csv.gz files |